# Molecular Dynamics of Lennard-Jones charged particles — Verlet integrator

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook runs a simple **molecular dynamics (MD)** simulation of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction). Where the
companion *energy-minimization* notebooks only walk the system **downhill** to a nearby
minimum, MD instead **integrates Newton's equations of motion** in time: the particles are
given initial velocities drawn from a Maxwell-Boltzmann distribution at a chosen temperature,
and then move under the forces they exert on one another. The system therefore *explores* the
energy surface — climbing barriers and sampling many configurations — rather than freezing
into the first minimum it finds.

## Theory in brief

### Lennard-Jones
Written from the squared distance $r^2$, with $Z = (r_{min}^2 / r^2)^3 = (r_{min}/r)^6$:

$$E_{LJ} = \varepsilon\, Z (Z-1)$$

The $Z^2$ term is the steep short-range **repulsion** (overlapping atoms), the $-Z$ term the
weaker long-range **attraction**. The two balance at $r = r_{min}$, where the energy reaches
its minimum $-\varepsilon$: `Epsilon` sets the well *depth* and `Rmin` its *position*.

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### Verlet integration
The **force** on each atom is minus the gradient of the total potential energy. Given the
force, Newton's law $\mathbf{F} = m\,\mathbf{a}$ is integrated with the **Verlet** algorithm,
which propagates positions using the current and *previous* positions:

$$\mathbf{r}(t+h) = 2\,\mathbf{r}(t) - \mathbf{r}(t-h) + \frac{\mathbf{F}(t)}{m}\,h^2$$

The very first step has no previous position, so it is bootstrapped from the initial
velocities with a Taylor expansion
$\mathbf{r}(h) = \mathbf{r}(0) + h\,\mathbf{v}(0) + \tfrac12 h^2 \mathbf{F}(0)/m$.
Velocities are recovered by central difference,
$\mathbf{v}(t) = [\mathbf{r}(t+h) - \mathbf{r}(t-h)] / (2h)$, and give the **kinetic energy**
$E_{kin} = \tfrac12 m \sum v_i^2$ and hence the instantaneous **temperature**
($E_{kin} = N k_B T$ in 2D). In a good simulation the **total** energy $E_{tot}=E_{pot}+E_{kin}$
is conserved while potential and kinetic energy flow back and forth; the time step `timestep`
must be small enough for this to hold.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import sqrt, log, sin, cos

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, randint, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions).
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy and temperature functions

The **potential** energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).
Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids a `sqrt` for every pair and lets the cutoff test (`distsquare < cutoffsquare`)
skip distant pairs cheaply. A `sqrt` is taken only where a term actually needs $r$ (Coulomb).

`calc_temp` turns the velocities into the **kinetic** energy $E_{kin}=\tfrac12 m\sum v_i^2$
and the instantaneous temperature via $E_{kin}=N k_B T$.

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1/distsquare)**3 * rmin_exp6
    return epsilon * Z * (Z - 1)

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Total potential energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    rmin_exp6 = rmin**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, rmin_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

# Kinetic energy and instantaneous temperature from the velocities
def calc_temp(vel, nat, k, mass):
    v2 = 0.0
    for i in range(len(vel)):
        v2 = v2 + vel[i][0]**2 + vel[i][1]**2
    kin = 0.5*mass*v2          # kinetic energy = 1/2 m v^2
    temp = kin/(nat*k)         # N k T = kinetic energy
    return kin, temp

## 4. Force functions

The force on an atom is minus the gradient of the total potential energy. For the
Lennard-Jones term the component along $x$ is obtained by the chain rule,

$$F_x = -\frac{\partial E}{\partial x} = -\frac{\partial E}{\partial Z}\,
        \frac{\partial Z}{\partial r}\,\frac{\partial r}{\partial x},$$

which maps directly onto the code:

* `dedz` $= \partial E/\partial Z = \varepsilon\,(2Z-1)$
* `dzdr` $= \partial Z/\partial r = -6\,r_{min}^{6}/r^{7}$
* `drdx` $= x_i/r$, where `xi` $= x_j - x_i$ — using the $j-i$ difference already carries the
  sign that turns $-\partial E/\partial x_i$ into the force on atom $i$.

The Coulomb force follows the same pattern, with `dedr` $= -q_a q_b/(\epsilon_r\, r^{2})$.

In [ ]:
# LJ force component (uses squared distance)
def ForceLJ2(distsquare, epsilon, rmin_exp6, xi):
    rij = sqrt(distsquare)
    Z = (1/distsquare)**3 * rmin_exp6
    dedz = epsilon*(2*Z - 1)
    dzdr = rmin_exp6*(-6.0/rij**7.0)
    drdx = xi/rij
    return dedz*dzdr*drdx

# Coulomb force component (uses squared distance)
def ForceCoulomb2(distsquare, dielec, qa, qb, xi):
    rij = sqrt(distsquare)
    dedr = -1.0*(qa*qb/dielec)*(1/distsquare)
    drdx = xi/rij
    return dedr*drdx

# Total force on each atom from Evdw + Ecoulomb (uses squared distance)
def Calc_Force2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim):
    Force = []
    rmin_exp6 = rmin**6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                fflist = []
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(distsquare, epsilon, rmin_exp6, tmp)
                    ff += ForceCoulomb2(distsquare, dielec, qa, qb, tmp)
                    fflist.append(ff)
                for k in range(2):
                    tmpforce[k] = tmpforce[k] + fflist[k]
        Force.append(tmpforce)
    return Force

## 5. The Verlet integrator

This is the core of the MD engine — the counterpart of the *minimizer* in the EM notebooks.
Three small functions carry out the time integration:

* **`Verlet`** — the workhorse. Given the current positions, the previous positions, the
  forces and the time step `h`, it produces the next positions with
  $\mathbf{r}(t+h) = 2\,\mathbf{r}(t) - \mathbf{r}(t-h) + \mathbf{F}\,h^2/m$.
* **`Step1`** — the bootstrap for the very first step (no previous positions yet): a Taylor
  expansion using the initial velocities,
  $\mathbf{r}(h) = \mathbf{r}(0) + h\,\mathbf{v}(0) + \tfrac12 h^2\mathbf{F}(0)/m$.
* **`CalcVel`** — recovers the velocities by central difference from the old and new
  positions, $\mathbf{v}(t) = [\mathbf{r}(t+h)-\mathbf{r}(t-h)]/(2h)$; these feed the kinetic
  energy and temperature.

In [ ]:
# One Verlet step: r(t+h) = 2 r(t) - r(t-h) + F/m * h^2
def Verlet(atom_coord, force, h, old_atom_coord, mass):
    newlist = []
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        oldr0x = old_atom_coord[i][0]
        oldr0y = old_atom_coord[i][1]
        r0xnew = 2*r0x - oldr0x + force[i][0]/mass * h**2
        r0ynew = 2*r0y - oldr0y + force[i][1]/mass * h**2
        newlist.append([r0xnew, r0ynew, q])
    return newlist

# First MD step (bootstrap from the initial velocities)
def Step1(atom_coord, velocity, force, h, mass):
    newlist = []
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        v0x = velocity[i][0]
        v0y = velocity[i][1]
        r0x = r0x + h*v0x + 0.5*h**2*force[i][0]/mass
        r0y = r0y + h*v0y + 0.5*h**2*force[i][1]/mass
        newlist.append([r0x, r0y, q])
    return newlist

# Velocities by central difference from the old and new positions
def CalcVel(old_atom_coord, atom_coord, h):
    newlist = []
    for i in range(len(atom_coord)):
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        oldr0x = old_atom_coord[i][0]
        oldr0y = old_atom_coord[i][1]
        v0x = (r0x - oldr0x)/(2*h)
        v0y = (r0y - oldr0y)/(2*h)
        newlist.append([v0x, v0y])
    return newlist

## 6. Parameters

These are the same parameters exposed by the sliders/entry boxes of the original GUI.
Change any of them and re-run **this cell together with the Initialisation and Run cells just
below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the toy
model's arbitrary units.

> **Note.** The **system** parameters below are the *same* as in the energy-minimization
> notebooks (`nAtoms=20`, `Radius=25`, `Epsilon=25`, `qat=Radius`, `Seed=100`, …), so the MD
> starts from the identical configuration those notebooks minimize. MD adds the extra
> **dynamical** parameters `Mass`, `Temperature` and `timestep`.

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms |
| `Mass` | particle mass (enters $\mathbf{a}=\mathbf{F}/m$) | 10 |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth | 1–100 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Molecular dynamics controls

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `Temperature` | target temperature for the initial velocities (K) | 300 |
| `timestep` | integration time step `h` (smaller → better energy conservation) | 5e-3 |
| `cstboltz` | Boltzmann constant (unit conversion) | fixed |
| `Seed` | random-number seed (reproducibility) | 100 |
| `nsteps` | number of MD steps to integrate | 2000 |

In [ ]:
nAtoms  = 20              # number of particles
Radius  = 25.0            # particle radius (must leave room to place all atoms)
Mass    = 10.0            # particle mass
Rmin    = 2.24 * Radius   # distance at which the LJ energy is minimal
BoxDim  = [500, 500]      # box dimensions
Epsilon = 25.0            # LJ well depth
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- molecular dynamics controls ---
cstboltz = 0.00198722          # Boltzmann constant in kcal/mol/K
cstboltz = 1000*cstboltz/4.18  # -> kJ/mol/K
Temperature = 300.0            # target temperature (K) for the initial velocities
timestep = 5.0e-3              # integration time step h
Seed     = 100                 # random number seed (reproducibility, shared with the EM notebooks)
nsteps   = 2000                # number of MD steps to run

## 7. Initialisation

Generate random, non-overlapping starting positions (a fraction `frac_neg` negative, the rest
positive), then draw **initial velocities** from a Maxwell-Boltzmann distribution at
`Temperature`. The overall (centre-of-mass) motion is removed and the velocities are rescaled
so the initial temperature is exactly the target value.

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom
    x = random()*(dim[0]-2*radius) + radius
    y = random()*(dim[1]-2*radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    if n == 2:
        tmp_coord.append([175, 300, charge])
    else:
        tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = -qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 10**10:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


### draw initial velocities from a Maxwell-Boltzmann distribution ###
def InitVel(n, temperature, cstboltz, mass):
    stdev = sqrt(cstboltz*temperature/mass)
    print("Initializing velocities, please wait...")
    tmp_vel = []
    if n == 2:                       # for testing: two atoms start at rest
        tmp_vel = [[0.0, 0.0], [0.0, 0.0]]
    else:
        for i in range(n):
            r1 = random()
            r2 = random()
            # Gaussian-distributed components (Box-Muller-like)
            x1 = sqrt(-2.0*log(r1))*cos(r2)
            x2 = sqrt(-2.0*log(r1))*sin(0.5*r2)
            tmp_vel.append([x1*stdev, x2*stdev])

    # remove overall (centre-of-mass) motion
    vxt = sum(v[0] for v in tmp_vel)
    vyt = sum(v[1] for v in tmp_vel)
    for i in range(n):
        tmp_vel[i][0] -= vxt/float(n)
        tmp_vel[i][1] -= vyt/float(n)

    # rescale so the temperature is exactly the target value
    kin, tt = calc_temp(tmp_vel, n, cstboltz, mass)
    scaling = sqrt(temperature/tt)
    vel = [[v[0]*scaling, v[1]*scaling] for v in tmp_vel]
    return vel


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Velocity   = InitVel(nAtoms, Temperature, cstboltz, Mass)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms and initialised their velocities.")

## 8. Run the molecular dynamics

This loop replaces the GUI's `Go` callback / Tkinter event loop. The first step is
bootstrapped with `Step1`; every step after that applies the `Verlet` update, recovers the
velocities with `CalcVel`, and records the potential, kinetic and total energies, the
temperature and a snapshot of the positions.

Periodic boundary conditions are applied by wrapping each just-moved position **together with
its immediate predecessor**, so the position *difference* that Verlet relies on is preserved
across the box edge (velocities are computed *before* wrapping, to avoid spurious jumps). The
run aborts early if the system "explodes" (temperature blowing up), which would signal a time
step that is too large.

In [ ]:
def run_md(Atom_Coord, Velocity):
    """Headless velocity-Verlet MD. Returns trajectory + energy/temperature histories."""
    coord = [list(a) for a in Atom_Coord]
    vel   = [list(v) for v in Velocity]
    h = timestep

    def pbc_pair(a, b):
        """Wrap position `a` into the box, applying the same shift to its predecessor `b`."""
        for pp in range(len(a)):
            for i in range(2):
                if a[pp][i] < 0:
                    a[pp][i] += BoxDim[i]; b[pp][i] += BoxDim[i]
                if a[pp][i] > BoxDim[i]:
                    a[pp][i] -= BoxDim[i]; b[pp][i] -= BoxDim[i]

    # --- first step: Taylor bootstrap from the initial velocities ---
    Force = Calc_Force2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
    old = [list(a) for a in coord]
    new = Step1(coord, vel, Force, h, Mass)     # r(h)

    # record step 0 (initial positions and velocities)
    Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
    Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)
    traj   = [[list(a) for a in coord]]
    Epot   = [Ene];  Elj = [EneLJ];  Ecoul = [EneCoul]
    Ekin   = [Kin];  Etot = [Ene + Kin];  Temp = [T]

    print("step %6d  t=%8.3f  Etot=%8.1f  Ekin=%7.1f  Epot=%7.1f  T=%6.1f" % (0, 0.0, Ene+Kin, Kin, Ene, T))

    pbc_pair(new, old)
    oldp = old
    coord = new

    for step in range(1, nsteps+1):
        Force = Calc_Force2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
        nw  = Verlet(coord, Force, h, oldp, Mass)   # next positions
        vel = CalcVel(oldp, nw, h)                  # velocity at the current step

        Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
        Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)

        traj.append([list(a) for a in coord])
        Epot.append(Ene); Elj.append(EneLJ); Ecoul.append(EneCoul)
        Ekin.append(Kin); Etot.append(Ene + Kin); Temp.append(T)

        if T > 1.0e6:
            print("The system is exploding -- emergency stop at step", step)
            break

        pbc_pair(nw, coord)     # wrap next positions together with the current ones
        oldp = coord
        coord = nw

        if step % 100 == 0:
            print("step %6d  t=%8.3f  Etot=%8.1f  Ekin=%7.1f  Epot=%7.1f  T=%6.1f"
                  % (step, step*h, Ene+Kin, Kin, Ene, T))

    time = [i*h for i in range(len(traj))]
    return traj, time, Epot, Elj, Ecoul, Ekin, Etot, Temp


traj, time, Epot, Elj, Ecoul, Ekin, Etot, Temp = run_md(Atom_Coord, Velocity)
print(f"\nDone: {len(traj)-1} MD steps.  Mean temperature ~ {sum(Temp)/len(Temp):.1f} K")

## 9. Energy and temperature

The left panel shows how the **potential**, **kinetic** and **total** energies evolve. Kinetic
and potential energy exchange continuously as atoms speed up and slow down, while the **total**
energy stays essentially flat — the signature of a well-behaved (energy-conserving) integrator.
The right panel shows the instantaneous temperature fluctuating around its mean.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(time, Etot, label="E$_{tot}$", lw=2)
ax1.plot(time, Ekin, label="E$_{kin}$", lw=1.3)
ax1.plot(time, Epot, label="E$_{pot}$", lw=1.3)
ax1.set_xlabel("time")
ax1.set_ylabel("energy")
ax1.set_title("Energy vs time (Verlet MD)")
ax1.legend()
ax1.grid(alpha=0.3)

Tmean = sum(Temp)/len(Temp)
ax2.plot(time, Temp, lw=1.0, color="tab:red")
ax2.axhline(Tmean, ls="--", color="black", lw=1, label=f"mean = {Tmean:.0f} K")
ax2.set_xlabel("time")
ax2.set_ylabel("temperature (K)")
ax2.set_title("Instantaneous temperature")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (t = {time[0]:.2f})")
draw_config(ax2, traj[-1], f"Final  (t = {time[-1]:.2f})")
plt.tight_layout()
plt.show()

## 11. Animation of the trajectory

Replays the whole trajectory. This reproduces the live view of the original GUI.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   t = {time[f]:.2f}   E$_{{tot}}$ = {Etot[f]:.0f}   T = {Temp[f]:.0f} K")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 12. Molecular dynamics vs energy minimization

This notebook **integrates the equations of motion**; the companion notebooks
(`LJ-ELEC_EM-steepest`, `LJ-ELEC_EM-conjugate`, `LJ-ELEC_EM-simplex`) instead **minimize** the
energy of the same kind of system. The key differences:

* **Energy minimization** only ever moves *downhill* and halts at the first nearby local
  minimum — it answers *"what is a stable arrangement close to where I started?"*.
* **Molecular dynamics** gives the particles kinetic energy, so they can climb **over energy
  barriers** and keep exploring. It conserves the **total** energy (potential + kinetic) rather
  than driving it to a minimum, and it produces a **time-dependent trajectory** from which one
  can measure temperature, diffusion, and thermodynamic averages.
* Because MD samples many configurations at finite temperature, it does **not** get permanently
  trapped in the first minimum — cooling an MD run down (lowering `Temperature`) is in fact one
  simple way to search for deeper minima than a single minimization would find.

Try lowering `Temperature`, shrinking `timestep` (better energy conservation), or increasing
`nsteps`, and re-run the Parameters → Initialisation → Run cells to see the effect.